In [2]:
import os
from os.path import exists
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import math
import copy
import time
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd
import altair as alt
from torchtext.data.functional import to_map_style_dataset
from torch.utils.data import DataLoader
from torchtext.vocab import build_vocab_from_iterator
import torchtext.datasets as datasets
import spacy
import GPUtil
import warnings
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP


# Set to False to skip notebook execution (e.g. for debugging)
warnings.filterwarnings("ignore")
RUN_EXAMPLES = True


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\aisys\Desktop\my_project_folder\Python_3.12\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\aisys\Desktop\my_project_folder\Python_3.12\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\aisys\Desktop\my_project_folder\Python_3.12\.venv\Lib\site-packages\ipykernel\ker

In [20]:
class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture. Base for this and many
    other models.
    """

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        # src: 소스 문장 토큰 ID 텐서
        self.src_embed = src_embed  # 인코더에서 패딩을 가리는 마스크 (GPU 연산을 위해 길이를 맞추려고 <pad>를 채움.)
        # tgt: 타깃 문장 디코더 입력 토큰 ID 텐서 (teacher forcing)
        self.tgt_embed = tgt_embed  # 디코더에서 쓰는 마스크
        self.generator = generator
        
        # 토큰 ID 텐서: 문장을 숫자로 변환한 결과. [batch, seq_len] 형태의 정수 텐서로, 임베딩 층에서 벡터로 바뀐다.


    def forward(self, src, tgt, src_mask, tgt_mask):
        # forward의 반환값의 두가지 패턴
        # 디코더의 마지막 hidden state → “forward는 중간 결과까지만, 학습 루프에서 후처리” 패턴.
        # logits (어휘 분포) -> forward 안에서 generator까지 호출. -> → “forward에서 최종 출력까지” 패턴.
        "Take in and process masked src and target sequences."
        return self.decoder(self.encode(src, src_mask), src_mask, tgt, tgt_mask)
    
    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)
    
    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)
    

In [21]:
# generator: 마지막 히든 벡터 → 어휘 확률 분포로 바꿔주는 모듈
class Generator(nn.Module):
    "Define standard linear + softmax generation step."

    def __init__(self, d_model, vocab):
        # d_model: Transformer의 hidden dimension (예: 512, 768 …).
        # vocab: 단어 집합 크기 (예: 30000).
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)  
        # nn.Linear(in_features, out_features)
        # d_model 차원(디코더 히든 벡터)을 어휘 크기(vocab) 차원으로 바꾼다
    
    def forward(self, x):
        # 입력 x: [B, T, d_model] (디코더 출력)
        return log_softmax(self.proj(x), dim=-1)
        # dim=-1 → 마지막 차원(vocab) 기준으로 softmax.→ 각 시점마다 어휘 분포(합=1) 계산.
        # [B, T, vocab]의 vocab 축을 의미.
        """
            batch: 몇 개 문장을 동시에 처리 중인가

            time: 시퀀스 길이 (토큰 수)

            vocab: 어휘 크기 (단어 후보 수)
        """

In [22]:
def clones(module, N):
    "Produce N identical layers."
    # N개의 모듈을 복사
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [23]:
class LayerNorm(nn.Module):
     "Construct a layernorm module (See citation for details)."

     def __init__(self, features, eps=1e-6) -> None:
          # featurds는 입력 텐서 x의 마지막 차원 크기 (= hidden size, d_model)를 의미.
          super().__init__()
          self.a_2 = nn.Parameter(torch.ones(features))     # scale (γ)
          self.b_2 = nn.Parameter(torch.zeros(features))    # shift (β)
          # nn.Parameter로 감쌌으니까, 이 값들은 학습 중 업데이트됨
          self.eps = eps # 표준편차가 0이 됬을 때 에러가 나지 않도록 더함
     
     def forward(self, x):
          # x : 이 레이어가 받는 입력 텐서
          # -1 : 마지막 차원을 기준
          # keepdim=True : 브로드캐스팅이 가능하게 shape를 유지
          mean = x.mean(-1, keepdim=True)
          std = x.std(-1, keepdim=True)
          return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

In [24]:
class Encoder(nn.Module):
    "Core encoder is a stack of N layers"

    def __init__(self, layer, N) -> None:
        super().__init__()
        self.layers = clones(layer, N)      # 인코더 레이어 N개 쌓기
        # layer 안에는 보통 (1) Multi-Head Self-Attention, (2) FFN(포지션별 MLP), (3) 잔차/Norm으로 구성
        # expert를 선택하는 로직은 Encoder 바깥이 아니라 레이어 내부의 FFN 자리에 들어가는 MoE 블록 안에 존재
        self.norm = LayerNorm(layer.size)
    
    def forward(self, x, mask):
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:
            x = layer(x, mask)  # 각 layer 안에서 이미 자체적으로 Norm을 수행
        return self.norm(x)     # elf.norm(x)는 스택 전체를 통과한 뒤 추가로 한 번 더 정규화

In [25]:
class SublayerConnection(nn.Module):
    """
    A residual connection followed by a layer norm.
    Note for code simplicity the norm is first as opposed to last.
    """

    def __init__(self, size, dropout) -> None:
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer):
        "Apply residual connection to any sublayer with the same size."
        return x + self.dropout(sublayer(self.norm(x)))

In [40]:
class EncoderLayer(nn.Module):
    "Encoder is made up of self-attn and feed forward (defined below)"
    # Each layer has two sub-layers
    # multi-head self-attention
    # feed-forward network

    def __init__(self, size, self_attn, feed_forward, dropout) -> None:
        super().__init__()
        self.self_attn = self_attn                  # multi-head self-attention 모듈
        self.feed_forward = feed_forward            # position-wise FFN 모듈
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size
        # sublayer[0] : Self-Attention + Residual + Norm
        # sublayer[1] : FFN + Residual + Norm
    
    def forward(self, x, mask):
         "Follow Figure 1 (left) for connections."
         # 첫 번째 sublayer: Self-Attention
         x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask)) # query, key, value를 입력으로 받음
         # 두 번째 sublayer: FFN
         return self.sublayer[1](x, self.feed_forward)

In [27]:
class Decoder(nn.Module):
    "Generic N layer decoder with masking."

    def __init__(self, layer, N) -> None:
        super().__init__()
        # 아미 기능(self-attention, FFN 등)이 들어 있는 하나의 레이어를 N개 독립 복제해서 층을 쌓는 것
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
    
    def forward(self, x, memory, src_mask, tgt_mask):
        # x = Decoder 입력
        # Encoder의 출력
        # src_mask: Encoder의 출력(memory)을 볼 때, 패딩 토큰(PAD) 같은 쓸모없는 부분은 보지 않도록 막음
        # 내부의 Self-Attention에서 쓰임
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

In [28]:
class DecoderLayer(nn.Module):
    "Decoder is made of self-attn, src-attn, and feed forward (defined below)"

    def __init__(self, size, self_attn, src_attn, feed_forward, dropout) -> None:
        super().__init__()
        self.size = size
        self.self_attn = self_attn  # 디코더 내부 self-attn
        self.src_attn = src_attn    # 인코더 출력(memory)을 보는 cross-attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)
    
    def forward(self, x, memory, src_mask, tgt_mask):
        "Follow Figure 1 (right) for connections."
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask)) # 디코더는 인코더의 K랑 V를 쓰고 Q만 조정
        return self.sublayer[2](x, self.feed_forward)

In [29]:
# (미래 단어 가리기 마스크)
def subsequent_mask(size):
    "Mask out subsequent positions."
    attn_shape = (1, size, size)    # Query 길이 × Key 길이 크기의 어텐션 점수 행렬 (QK^T)
    # Query / Key 길이 == 토큰 수
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(torch.uint8)
    # upper triangular을 생성
    # diagonal=0이면 주대각선 포함
    # diagonal=1이면 주대각선 바로 위부터 1이 남음
    return subsequent_mask == 0

In [30]:
def attention(query, key, value, mask=None, dropout=None):
    "Compute 'Scaled Dot Product Attention'"
    d_k = query.size(-1)
    # transpose(dim1, dim2)는 두 차원의 위치를 바꿔주는 함수
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
        # -1e9를 넣는 이유: softmax 연산에서 거의 확률 0이 되도록 강제하는 트릭
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

In [31]:
class MultiHeadedAttention(nn.Module):
    "Take in model size and number of heads."
    # h: 헤드 수
    # d_model: 모델의 임베딩 차원 수 (단어 임베딩 벡터나 attention hidden state의 크기)
    # 전체 벡터를 head별로 다른 projection으로 변환해서 작은 공간에서 attention을 계산
    def __init__(self, h, d_model, dropout=0.1) -> None:
        super().__init__()
        assert d_model % h == 0 # 조건식이 True면 pass, False면 프로그램이 즉시 멈추고 AssertionError
        self.d_k = d_model // h
        # [토큰 임베딩 512차원] → [8개 헤드: 64차원씩] → [concat → 다시 512차원]
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        # linears[0]: Query projection
        # linears[1]: Key projection  
        # linears[2]: Value projection
        # linears[3]: Output projection
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)
    
    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)

        # 1) Do all the linear projections in batch from d_model => h x d_k
        # 헤드별로 나누기
        query, key, value = [ lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2) 
                             for lin, x in zip(self.linears, (query, key, value))]
        # zip -> linear0과 query 짝 / linear1과 key 짝 / linear2과 value 짝
        # view: [B, T, d_model] -> [B, T, h, d_k] (헤드 수 h × 헤드 차원 d_k 로 쪼개기 / d_model = num_heads * d_k) 
        # transpose -> [B, h, T, d_k]

        # 2) Apply attention on all the projected vectors in batch.
        x, self.atten = attention(query, key, value, mask=mask, dropout=self.dropout)

        # 3) "Concat" using a view and apply a final linear.
        x = (
                x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)
        )
        # contiguous() — PyTorch의 메모리 연속성 보장
        # transpose같은 연산은 데이터를 실제로 복사하지 않고 “뷰(view)”만 바꿔서 보여줌 / contiguous()는 실제로 메모리를 새로 복사해서 연속된 배열로 재배치

        
        del query   # del은 변수 이름과 객체의 참조를 끊어주는 역할
        del key     # PyTorch는 GPU 메모리(CUDA 메모리)를 별도로 관리
        del value   # 즉, 파이썬 객체가 삭제돼야 GPU의 VRAM도 해제
        return self.linears[-1](x)    # 출력 통합용 Linear layer

In [32]:
class PositionwiseFeedForward(nn.Module):
    "Implements FFN equation."
    # 어텐션 이후에 개별 토큰 단위로 적용되는 비선형 변환층(FFN)
    def __init__(self, d_model, d_ff, dropout=0.1) -> None:
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff) # 512 -> 2048
        self.w_2 = nn.Linear(d_ff, d_model) # 2048 -> 512
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.w_2(self.dropout(self.w_1(x).relu()))

In [33]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab) -> None:
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model) # 단어 ID → 임베딩 벡터 변환 (lut은 lookup table의 약자)
        self.d_model = d_model
    
    def forward(self, x):
        # 임베딩 벡터의 분산(variance)을 1 근처로 맞추기 위해 곱함
        return self.lut(x) * math.sqrt(self.d_model)

In [38]:
class PositionalEncoding(nn.Module):
    "Implement the PE function."

    def __init__(self, d_model, dropout, max_len=5000) -> None:
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        # exp(-(2i/d_model) * log(10000)) -> 10000^-(2i/d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # [1, max_len, d_model]
        pe = pe.unsqueeze(0)
        # register_buffer → pe를 모델의 state로 저장하되, 학습되지 않게 등록.
        self.register_buffer('pe', pe)  # 모듈 수준에서 유지되는 정적(Static) 지역변수
    
    def forward(self, x):
        # x.size(1) = 현재 입력 문장의 길이
        # pe는 역전파(gradient) 계산에 포함하지 마라고 명시
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)

In [42]:
def make_model(src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1):
    "Helper: Construct a model from hyperparameters."
    # src_vocab	입력(소스 언어) 단어 사전 크기
    # tgt_vocab	출력(타겟 언어) 단어 사전 크기
    # N	인코더·디코더 층(layer) 수
    # d_model	단어 임베딩 및 내부 표현 차원
    # d_ff	FFN(피드포워드 네트워크)의 은닉 차원
    # h	Multi-head Attention의 헤드 수
    # dropout	dropout 비율
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab)
    )

    # This was important from their code.
    # Initialize parameters with Glorot / fan_avg.
    for p in model.parameters():
        if p.dim() > 1:
            # Xavier: 뉴런의 입력과 출력의 분산이 일정하게 유지되도록 가중치를 균등분포로 초기화하는 방법
            nn.init.xavier_uniform_(p)
    return model

In [43]:
def inference_test():
    test_model = make_model(11, 11, 2)
    test_model.eval()
    src = torch.LongTensor([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]])
    src_mask = torch.ones(1, 1, 10)

    memory = test_model.encode(src, src_mask)
    ys = torch.zeros(1, 1).type_as(src)

    for i in range(9):
        out = test_model.decode(memory, src_mask, ys, subsequent_mask(ys.size(1)).type_as(src.data))
        prob = test_model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat([ys, torch.empty(1, 1).type_as(src.data).fill_(next_word)], dim=1)
    
    print("Example Untrained Model Prediction:", ys)

def run_tests():
    for _ in range(10):
        inference_test()

run_tests()

Example Untrained Model Prediction: tensor([[0, 2, 6, 5, 5, 5, 5, 5, 5, 5]])
Example Untrained Model Prediction: tensor([[0, 3, 9, 9, 9, 2, 7, 0, 3, 9]])
Example Untrained Model Prediction: tensor([[0, 7, 3, 6, 7, 3, 0, 7, 3, 0]])
Example Untrained Model Prediction: tensor([[0, 0, 0, 0, 0, 6, 8, 8, 6, 8]])
Example Untrained Model Prediction: tensor([[0, 7, 0, 7, 0, 7, 0, 7, 0, 7]])
Example Untrained Model Prediction: tensor([[0, 0, 0, 0, 3, 0, 0, 0, 0, 0]])
Example Untrained Model Prediction: tensor([[ 0,  6,  6, 10,  3,  1,  9,  6,  1,  9]])
Example Untrained Model Prediction: tensor([[0, 5, 7, 7, 7, 4, 4, 4, 4, 4]])
Example Untrained Model Prediction: tensor([[0, 5, 5, 5, 5, 5, 5, 5, 5, 5]])
Example Untrained Model Prediction: tensor([[0, 4, 8, 3, 6, 8, 6, 8, 3, 6]])


In [ ]:
class Batch:
    """Object for holding a batch of data with mask during training."""
    # 입력 문장(src)
    # 출력 문장(tgt)
    # 인코더 마스크(src_mask)
    # 디코더 마스크(tgt_mask)
    # 실제 정답(tgt_y)
    # 유효 토큰 수(ntokens)
    def __init__(self, src, tgt=None, pad=2) -> None:
        self.src = src  # 0 또는 2가 PAD 토큰 역할 / 단어 ID가 2인 건 “<blank>”
        self.src_mask = (src != pad).unsqueeze(-2)  # PAD가 아닌 위치는 True, PAD는 False
        if tgt is not None:
            self.tgt = tgt[:, :-1]
            self.tgt_y = tgt[:, 1:]
            self.tgt_mask = self.make_std_mask(self.make_std_mask(self.tgt, pad))
            self.ntokens = (self.tgt_y != pad).data.sum()
    
    @staticmethod
    def make_std_mask(tgt, pad):
        "Create a mask to hide padding and future words."
        tgt_mask = (tgt != pad).unsqueeze(-2)
        tgt_mask  = tgt_mask & subsequent_mask(tgt.size(-1)).type_as(tgt_mask.data)
        return tgt_mask

In [ ]:
a = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])

print(a[:, :-1])
print()
print(a[:, 1:])

tensor([[1],
        [5],
        [9]])

tensor([[ 2,  3,  4],
        [ 6,  7,  8],
        [10, 11, 12]])
